# Phase 5: OCR & Document Intelligence
## Day 25: DocumentIntelligenceProject

Date: 2026-04-24

### Learning objectives
- Build an end-to-end document intelligence mini project.
- Create synthetic document images.
- Preprocess clean, noisy, and skewed documents.
- Extract OCR text with safe fallbacks.
- Convert OCR text into validated JSON records.
- Run quality checks and create a final dataset.

In [ ]:
import json
import re
import shutil
import textwrap
from pprint import pprint
from typing import Optional, Dict, Any, List

import numpy as np
import pandas as pd

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    cv2 = None
    CV2_AVAILABLE = False

try:
    from PIL import Image, ImageDraw, ImageFont, ImageFilter
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

try:
    import pytesseract
    PYTESSERACT_AVAILABLE = True
except Exception:
    pytesseract = None
    PYTESSERACT_AVAILABLE = False

try:
    from pydantic import BaseModel, Field
    PYDANTIC_AVAILABLE = True
except Exception:
    BaseModel = object
    Field = None
    PYDANTIC_AVAILABLE = False

TESSERACT_BINARY_AVAILABLE = shutil.which("tesseract") is not None

def show(title, content):
    print("\n" + "=" * 82)
    print(title)
    print("=" * 82)
    print(textwrap.dedent(str(content)).strip())

def model_to_dict(model):
    if hasattr(model, "model_dump"):
        return model.model_dump()
    if hasattr(model, "dict"):
        return model.dict()
    return model

print("Setup complete.")
print("OpenCV available:", CV2_AVAILABLE)
print("PIL available:", PIL_AVAILABLE)
print("pytesseract available:", PYTESSERACT_AVAILABLE)
print("Tesseract binary available:", TESSERACT_BINARY_AVAILABLE)
print("Pydantic available:", PYDANTIC_AVAILABLE)

In [ ]:
documents = [
    {
        "doc_id": "D001",
        "doc_type": "invoice",
        "condition": "clean",
        "text": '''
        DOCUMENT TYPE: INVOICE
        Invoice ID: INV-7101
        Vendor: Berlin Coffee Bar
        Date: 2026-04-24
        Campaign: Spring Coffee Push
        Channel: Instagram
        Spend: 1200 EUR
        Clicks: 3420
        Conversions: 184
        Total: 1200 EUR
        '''
    },
    {
        "doc_id": "D002",
        "doc_type": "receipt",
        "condition": "noisy",
        "text": '''
        DOCUMENT TYPE: RECEIPT
        Receipt ID: REC-2205
        Vendor: Berlin Bakery
        Date: 2026-04-23
        Item: Latte
        Item Total: 4.20 EUR
        Tax: 0.30 EUR
        Total: 4.50 EUR
        '''
    },
    {
        "doc_id": "D003",
        "doc_type": "invoice",
        "condition": "skewed",
        "text": '''
        DOCUMENT TYPE: INVOICE
        Invoice ID: INV-7102
        Vendor: Yoga Studio Berlin
        Date: 2026-04-22
        Campaign: Yoga Studio Trial
        Channel: TikTok
        Spend: 650 EUR
        Clicks: 2100
        Conversions: 165
        Total: 650 EUR
        '''
    },
    {
        "doc_id": "D004",
        "doc_type": "delivery_note",
        "condition": "clean",
        "text": '''
        DOCUMENT TYPE: DELIVERY NOTE
        Delivery ID: DEL-9010
        Vendor: Market Logistics
        Date: 2026-04-21
        City: Berlin
        Packages: 12
        Delayed Packages: 2
        Status: partial_delay
        '''
    },
    {
        "doc_id": "D005",
        "doc_type": "invoice",
        "condition": "noisy",
        "text": '''
        DOCUMENT TYPE: INVOICE
        Invoice ID: INV-7103
        Vendor: Bank App Team
        Date: 2026-04-20
        Campaign: Bank App Onboarding
        Channel: Email
        Spend: 800 EUR
        Clicks: 980
        Conversions: 42
        Total: 800 EUR
        '''
    }
]

target_schema = {
    "doc_id": "string",
    "doc_type": "invoice | receipt | delivery_note | unknown",
    "document_id": "string or null",
    "vendor": "string or null",
    "date": "YYYY-MM-DD string or null",
    "campaign": "string or null",
    "channel": "string or null",
    "spend_eur": "number or null",
    "clicks": "integer or null",
    "conversions": "integer or null",
    "item_total_eur": "number or null",
    "tax_eur": "number or null",
    "total_eur": "number or null",
    "city": "string or null",
    "packages": "integer or null",
    "delayed_packages": "integer or null",
    "status": "string or null"
}

print("Number of documents:", len(documents))
show("First document text", documents[0]["text"])
print("\nTarget schema:")
print(json.dumps(target_schema, indent=2))

## 1. Project goal

This mini project simulates a real document intelligence workflow.

The pipeline will create document images, preprocess them, run OCR, extract structured fields, validate the output, and build a clean dataset.

In [ ]:
project_steps = pd.DataFrame([
    {"step": 1, "stage": "Create images", "output": "Synthetic document images"},
    {"step": 2, "stage": "Preprocess", "output": "Cleaner images for OCR"},
    {"step": 3, "stage": "OCR", "output": "Raw text from images"},
    {"step": 4, "stage": "Clean OCR text", "output": "Less noisy text"},
    {"step": 5, "stage": "Extract JSON", "output": "Structured records"},
    {"step": 6, "stage": "Validate", "output": "Safe records or errors"},
    {"step": 7, "stage": "Analyze", "output": "Final dataset and quality report"},
])

project_steps

## 2. Create document images

Real projects start with PDFs, scans, or photos.

Here we generate document images from text so the notebook has no external file dependency.

In [ ]:
def create_document_image(text, width=920, height=520, font_size=23):
    if not PIL_AVAILABLE:
        return None

    image = Image.new("RGB", (width, height), color="white")
    draw = ImageDraw.Draw(image)

    try:
        font = ImageFont.truetype("DejaVuSansMono.ttf", font_size)
    except Exception:
        font = ImageFont.load_default()

    draw.multiline_text((40, 35), text.strip(), fill="black", font=font, spacing=10)
    return image

def pil_to_rgb_array(image):
    if image is None:
        return None
    return np.array(image.convert("RGB"))

def rgb_array_to_pil(array):
    if array is None or not PIL_AVAILABLE:
        return None
    return Image.fromarray(np.clip(array, 0, 255).astype(np.uint8))

def display_image(image_or_array):
    if image_or_array is None:
        print("No image to display.")
        return

    if not PIL_AVAILABLE:
        print("PIL is not available.")
        return

    if isinstance(image_or_array, Image.Image):
        display(image_or_array)
    else:
        array = np.clip(image_or_array, 0, 255).astype(np.uint8)
        if len(array.shape) == 2:
            display(Image.fromarray(array))
        else:
            display(rgb_array_to_pil(array))

sample_image = create_document_image(documents[0]["text"])
display_image(sample_image)

In [ ]:
def add_noise(image_rgb, noise_std=32, seed=42):
    if image_rgb is None:
        return None

    rng = np.random.default_rng(seed)
    noise = rng.normal(0, noise_std, image_rgb.shape)
    noisy = image_rgb.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def rotate_image(image_rgb, angle_degrees):
    if image_rgb is None:
        return None

    if CV2_AVAILABLE:
        height, width = image_rgb.shape[:2]
        center = (width // 2, height // 2)
        matrix = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)
        return cv2.warpAffine(image_rgb, matrix, (width, height), borderValue=(255, 255, 255))

    if PIL_AVAILABLE:
        pil = rgb_array_to_pil(image_rgb)
        return np.array(pil.rotate(angle_degrees, expand=False, fillcolor="white"))

    return image_rgb

def create_conditioned_image(doc, index=0):
    base = pil_to_rgb_array(create_document_image(doc["text"]))

    if doc["condition"] == "noisy":
        return add_noise(base, noise_std=34, seed=100 + index)

    if doc["condition"] == "skewed":
        return rotate_image(base, angle_degrees=6)

    return base

document_images = {
    doc["doc_id"]: create_conditioned_image(doc, index=i)
    for i, doc in enumerate(documents)
}

print("Created document images:", list(document_images.keys()))
display_image(document_images["D002"])

## 3. Preprocessing functions

Preprocessing prepares images for OCR.

We use grayscale, denoising, Otsu thresholding, adaptive thresholding, and optional deskewing.

In [ ]:
def to_grayscale(image_rgb):
    if image_rgb is None:
        return None

    if CV2_AVAILABLE:
        return cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

    return np.dot(image_rgb[..., :3], [0.299, 0.587, 0.114]).astype(np.uint8)

def denoise_image(gray_image):
    if gray_image is None:
        return None

    if CV2_AVAILABLE:
        return cv2.fastNlMeansDenoising(gray_image, None, h=18, templateWindowSize=7, searchWindowSize=21)

    if PIL_AVAILABLE:
        pil = Image.fromarray(gray_image.astype(np.uint8))
        return np.array(pil.filter(ImageFilter.MedianFilter(size=3)))

    return gray_image

def otsu_threshold(gray_image):
    if gray_image is None:
        return None, None

    if CV2_AVAILABLE:
        value, binary = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary, value

    threshold = int(np.mean(gray_image))
    binary = np.where(gray_image > threshold, 255, 0).astype(np.uint8)
    return binary, threshold

def adaptive_threshold(gray_image, block_size=35, c_value=11):
    if gray_image is None:
        return None

    if block_size % 2 == 0:
        block_size += 1

    if CV2_AVAILABLE:
        return cv2.adaptiveThreshold(
            gray_image,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            block_size,
            c_value
        )

    return np.where(gray_image > np.mean(gray_image), 255, 0).astype(np.uint8)

gray_example = to_grayscale(document_images["D001"])
binary_example, otsu_value = otsu_threshold(gray_example)

print("Otsu value:", otsu_value)
display_image(binary_example)

In [ ]:
def estimate_skew_angle(gray_image):
    if gray_image is None or not CV2_AVAILABLE:
        return 0.0

    _, binary = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(binary > 0))

    if len(coords) < 10:
        return 0.0

    angle = cv2.minAreaRect(coords)[-1]

    if angle < -45:
        angle = 90 + angle

    return float(angle)

def deskew_rgb(image_rgb):
    if image_rgb is None:
        return None, 0.0

    angle = estimate_skew_angle(to_grayscale(image_rgb))
    corrected = rotate_image(image_rgb, angle_degrees=angle)
    return corrected, angle

deskewed, angle = deskew_rgb(document_images["D003"])

print("Detected correction angle:", round(angle, 2))
display_image(deskewed)

In [ ]:
def preprocess_image(image_rgb, condition="clean"):
    if image_rgb is None:
        return {
            "image": None,
            "method": "none",
            "deskew_angle": 0.0,
            "threshold_value": None
        }

    working = image_rgb
    deskew_angle = 0.0

    if condition == "skewed":
        working, deskew_angle = deskew_rgb(working)

    gray = to_grayscale(working)

    if condition == "noisy":
        gray = denoise_image(gray)
        processed = adaptive_threshold(gray, block_size=35, c_value=11)
        method = "denoise + adaptive"
        threshold_value = "adaptive"
    else:
        gray = denoise_image(gray)
        processed, threshold_value = otsu_threshold(gray)
        method = "denoise + otsu"

    return {
        "image": processed,
        "method": method,
        "deskew_angle": deskew_angle,
        "threshold_value": threshold_value
    }

preprocessed = {
    doc["doc_id"]: preprocess_image(document_images[doc["doc_id"]], doc["condition"])
    for doc in documents
}

for doc_id, result in preprocessed.items():
    print(doc_id, {k: v for k, v in result.items() if k != "image"})

## 4. OCR with safe fallback

The notebook uses real Tesseract OCR when available.

If Tesseract is not installed, it uses the original document text as a mock OCR result.

In [ ]:
def find_doc(doc_id):
    for doc in documents:
        if doc["doc_id"] == doc_id:
            return doc
    raise ValueError(f"Unknown doc_id: {doc_id}")

def simulate_ocr_errors(text, condition):
    if condition != "noisy":
        return text.strip()

    replacements = {
        "DOCUMENT TYPE": "D0CUMENT TYPE",
        "Invoice": "lnvoice",
        "Receipt": "Rece1pt",
        "Spend": "5pend",
        "Clicks": "C1icks",
        "Conversions": "Convers1ons",
        "Total": "TotaI",
        "Item Total": "Item TotaI",
        "Packages": "Package5",
    }

    noisy = text.strip()
    for good, bad in replacements.items():
        noisy = noisy.replace(good, bad)
    return noisy

def run_ocr(preprocessed_image, doc_id, lang="eng", config="--psm 6 --oem 3"):
    doc = find_doc(doc_id)

    if preprocessed_image is not None and PYTESSERACT_AVAILABLE and TESSERACT_BINARY_AVAILABLE and PIL_AVAILABLE:
        try:
            pil_image = Image.fromarray(preprocessed_image.astype(np.uint8))
            return pytesseract.image_to_string(pil_image, lang=lang, config=config)
        except Exception as error:
            print(f"Real OCR failed for {doc_id}. Using mock OCR.")
            print("Error:", error)

    return simulate_ocr_errors(doc["text"], doc["condition"])

ocr_outputs = {
    doc["doc_id"]: run_ocr(preprocessed[doc["doc_id"]]["image"], doc["doc_id"])
    for doc in documents
}

show("OCR output for noisy receipt", ocr_outputs["D002"])

## 5. Clean OCR text

OCR output can have spacing issues and common character mistakes.

A cleanup function makes extraction more reliable.

In [ ]:
def clean_ocr_text(text):
    text = text.replace("\x0c", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

def repair_common_ocr_errors(text):
    replacements = {
        "D0CUMENT TYPE": "DOCUMENT TYPE",
        "lnvoice": "Invoice",
        "Rece1pt": "Receipt",
        "5pend": "Spend",
        "C1icks": "Clicks",
        "Convers1ons": "Conversions",
        "TotaI": "Total",
        "Item Total": "Item Total",
        "Item TotaI": "Item Total",
        "Package5": "Packages",
    }

    repaired = clean_ocr_text(text)
    for wrong, correct in replacements.items():
        repaired = repaired.replace(wrong, correct)
    return repaired

cleaned_ocr_outputs = {
    doc_id: repair_common_ocr_errors(text)
    for doc_id, text in ocr_outputs.items()
}

show("Cleaned OCR for D002", cleaned_ocr_outputs["D002"])

In [ ]:
def text_quality_score(text):
    repaired = repair_common_ocr_errors(text)
    important_keywords = [
        "DOCUMENT TYPE", "Date", "Vendor", "Total",
        "Invoice ID", "Receipt ID", "Delivery ID",
        "Campaign", "Channel", "Spend", "Clicks", "Conversions",
        "Packages", "Status"
    ]

    found = [keyword for keyword in important_keywords if keyword in repaired]
    return {
        "found_count": len(found),
        "possible_count": len(important_keywords),
        "score": round(len(found) / len(important_keywords), 3),
        "found_keywords": found
    }

quality_rows = []

for doc_id, text in ocr_outputs.items():
    score = text_quality_score(text)
    quality_rows.append({
        "doc_id": doc_id,
        "quality_score": score["score"],
        "found_count": score["found_count"]
    })

pd.DataFrame(quality_rows)

## 6. Build the extraction prompt

The extraction prompt gives the OCR text and the target schema.

The model should return valid JSON only.

In [ ]:
def build_extraction_prompt(doc_id, ocr_text, schema):
    return f'''
Task:
Extract structured fields from OCR text.

Document ID:
{doc_id}

OCR text:
{repair_common_ocr_errors(ocr_text)}

Rules:
- Return valid JSON only.
- Do not include markdown.
- Do not include explanations.
- Use null for missing fields.
- Numeric fields must be numbers.
- Use this schema exactly.

JSON schema:
{json.dumps(schema, indent=2)}
'''.strip()

prompt = build_extraction_prompt("D001", ocr_outputs["D001"], target_schema)
show("Extraction prompt", prompt)

## 7. Mock LLM extractor

In a real app, this step would call OpenAI, Ollama, or another model.

Here we use regex as a mock LLM so the project runs locally.

In [ ]:
def get_field(pattern, text, flags=re.IGNORECASE):
    match = re.search(pattern, text, flags=flags)
    return match.group(1).strip() if match else None

def get_int(pattern, text):
    value = get_field(pattern, text)
    return int(value) if value is not None else None

def get_float(pattern, text):
    value = get_field(pattern, text)
    return float(value) if value is not None else None

def infer_doc_type(text):
    text_upper = text.upper()
    if "INVOICE" in text_upper:
        return "invoice"
    if "RECEIPT" in text_upper:
        return "receipt"
    if "DELIVERY NOTE" in text_upper:
        return "delivery_note"
    return "unknown"

def get_document_id(text, doc_type):
    if doc_type == "invoice":
        return get_field(r"Invoice ID:\s*([A-Z]+-[0-9]+)", text)
    if doc_type == "receipt":
        return get_field(r"Receipt ID:\s*([A-Z]+-[0-9]+)", text)
    if doc_type == "delivery_note":
        return get_field(r"Delivery ID:\s*([A-Z]+-[0-9]+)", text)
    return None

def mock_llm_extract(doc_id, ocr_text):
    text = repair_common_ocr_errors(ocr_text)
    doc_type = infer_doc_type(text)

    record = {
        "doc_id": doc_id,
        "doc_type": doc_type,
        "document_id": get_document_id(text, doc_type),
        "vendor": get_field(r"Vendor:\s*(.+)", text),
        "date": get_field(r"Date:\s*(\d{4}-\d{2}-\d{2})", text),
        "campaign": get_field(r"Campaign:\s*(.+)", text),
        "channel": get_field(r"Channel:\s*([A-Za-z]+)", text),
        "spend_eur": get_float(r"Spend:\s*([0-9]+(?:\.[0-9]+)?)\s*EUR", text),
        "clicks": get_int(r"Clicks:\s*(\d+)", text),
        "conversions": get_int(r"Conversions:\s*(\d+)", text),
        "item_total_eur": get_float(r"Item Total:\s*([0-9]+(?:\.[0-9]+)?)\s*EUR", text),
        "tax_eur": get_float(r"Tax:\s*([0-9]+(?:\.[0-9]+)?)\s*EUR", text),
        "total_eur": get_float(r"Total:\s*([0-9]+(?:\.[0-9]+)?)\s*EUR", text),
        "city": get_field(r"City:\s*([A-Za-z\s]+)", text),
        "packages": get_int(r"Packages:\s*(\d+)", text),
        "delayed_packages": get_int(r"Delayed Packages:\s*(\d+)", text),
        "status": get_field(r"Status:\s*([A-Za-z_]+)", text),
    }

    return json.dumps(record, indent=2)

llm_output = mock_llm_extract("D001", ocr_outputs["D001"])
print(llm_output)

## 8. Parse and repair JSON

LLM output can contain markdown, prose, or trailing commas.

The parser should repair cheap formatting issues and fail clearly when needed.

In [ ]:
def safe_json_loads(text):
    try:
        return json.loads(text), None
    except json.JSONDecodeError as error:
        return None, str(error)

def extract_json_block(text):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    return match.group(0) if match else None

def repair_json_text(text):
    repaired = text.strip()
    repaired = repaired.replace("```json", "").replace("```", "").strip()
    repaired = re.sub(r",\s*}", "}", repaired)
    repaired = re.sub(r",\s*]", "]", repaired)
    return repaired

messy_json = '''
Here is the result:
```json
{
  "doc_id": "D001",
  "doc_type": "invoice",
  "document_id": "INV-7101",
}
```
'''

block = extract_json_block(messy_json)
repaired = repair_json_text(block)
data, error = safe_json_loads(repaired)

pprint(data)
print("Error:", error)

## 9. Validate extracted records

Validation protects the final dataset.

It checks required fields, types, allowed document types, dates, and simple business rules.

In [ ]:
allowed_doc_types = {"invoice", "receipt", "delivery_note", "unknown"}

if PYDANTIC_AVAILABLE:
    class ExtractedDocument(BaseModel):
        doc_id: str
        doc_type: str
        document_id: Optional[str] = None
        vendor: Optional[str] = None
        date: Optional[str] = None
        campaign: Optional[str] = None
        channel: Optional[str] = None
        spend_eur: Optional[float] = Field(default=None, ge=0)
        clicks: Optional[int] = Field(default=None, ge=0)
        conversions: Optional[int] = Field(default=None, ge=0)
        item_total_eur: Optional[float] = Field(default=None, ge=0)
        tax_eur: Optional[float] = Field(default=None, ge=0)
        total_eur: Optional[float] = Field(default=None, ge=0)
        city: Optional[str] = None
        packages: Optional[int] = Field(default=None, ge=0)
        delayed_packages: Optional[int] = Field(default=None, ge=0)
        status: Optional[str] = None
else:
    ExtractedDocument = None

def fallback_validate_record(data):
    required_fields = [
        "doc_id", "doc_type", "document_id", "vendor", "date",
        "campaign", "channel", "spend_eur", "clicks", "conversions",
        "item_total_eur", "tax_eur", "total_eur", "city",
        "packages", "delayed_packages", "status"
    ]

    errors = []

    for field in required_fields:
        if field not in data:
            errors.append(f"Missing field: {field}")

    if data.get("doc_type") not in allowed_doc_types:
        errors.append("doc_type is not allowed")

    if data.get("date") is not None and not re.match(r"^\d{4}-\d{2}-\d{2}$", data["date"]):
        errors.append("date must use YYYY-MM-DD")

    numeric_fields = [
        "spend_eur", "clicks", "conversions", "item_total_eur",
        "tax_eur", "total_eur", "packages", "delayed_packages"
    ]

    for field in numeric_fields:
        value = data.get(field)
        if value is not None and not isinstance(value, (int, float)):
            errors.append(f"{field} must be numeric or null")
        if isinstance(value, (int, float)) and value < 0:
            errors.append(f"{field} must be non-negative")

    if data.get("clicks") is not None and data.get("conversions") is not None:
        if data["conversions"] > data["clicks"]:
            errors.append("conversions cannot be greater than clicks")

    if data.get("packages") is not None and data.get("delayed_packages") is not None:
        if data["delayed_packages"] > data["packages"]:
            errors.append("delayed_packages cannot be greater than packages")

    if errors:
        raise ValueError(errors)

    return data

def validate_record(data):
    if PYDANTIC_AVAILABLE:
        model = ExtractedDocument(**data)
        model_data = model_to_dict(model)
        fallback_validate_record(model_data)
        return model
    return fallback_validate_record(data)

parsed_data, error = safe_json_loads(llm_output)
validated = validate_record(parsed_data)

pprint(model_to_dict(validated))

In [ ]:
invalid_example = {
    "doc_id": "D999",
    "doc_type": "invoice",
    "document_id": "INV-9999",
    "vendor": "Bad Data Vendor",
    "date": "24-04-2026",
    "campaign": "Bad Campaign",
    "channel": "Email",
    "spend_eur": -100,
    "clicks": 10,
    "conversions": 20,
    "item_total_eur": None,
    "tax_eur": None,
    "total_eur": 100,
    "city": None,
    "packages": None,
    "delayed_packages": None,
    "status": None
}

try:
    validate_record(invalid_example)
except Exception as error:
    print("Validation failed:")
    print(error)

## 10. Build the full project pipeline

Now we connect every step into one function.

Each result stores metadata, OCR text, extracted record, and errors.

In [ ]:
def parse_and_validate_llm_output(llm_output):
    json_text = extract_json_block(llm_output) or llm_output
    json_text = repair_json_text(json_text)

    data, parse_error = safe_json_loads(json_text)

    if parse_error:
        return None, f"Parse error: {parse_error}"

    try:
        validated = validate_record(data)
        return model_to_dict(validated), None
    except Exception as validation_error:
        return None, f"Validation error: {validation_error}"

def make_retry_prompt(doc_id, ocr_text, bad_output, error_message):
    return f'''
The previous extraction failed.

Document ID:
{doc_id}

OCR text:
{repair_common_ocr_errors(ocr_text)}

Bad output:
{bad_output}

Error:
{error_message}

Return corrected valid JSON only using this schema:
{json.dumps(target_schema, indent=2)}
'''.strip()

def run_document_pipeline(doc, simulate_broken_json=False):
    doc_id = doc["doc_id"]

    preprocess_result = preprocess_image(
        document_images[doc_id],
        condition=doc["condition"]
    )

    ocr_text = run_ocr(preprocess_result["image"], doc_id)
    cleaned_text = repair_common_ocr_errors(ocr_text)
    quality = text_quality_score(ocr_text)
    prompt = build_extraction_prompt(doc_id, ocr_text, target_schema)

    llm_output = mock_llm_extract(doc_id, cleaned_text)

    if simulate_broken_json:
        llm_output = llm_output[:-2] + ",\n}"

    record, error = parse_and_validate_llm_output(llm_output)
    retry_used = False
    retry_prompt = None

    if error:
        retry_used = True
        retry_prompt = make_retry_prompt(doc_id, cleaned_text, llm_output, error)
        retry_output = mock_llm_extract(doc_id, cleaned_text)
        record, error = parse_and_validate_llm_output(retry_output)
    else:
        retry_output = None

    return {
        "doc_id": doc_id,
        "condition": doc["condition"],
        "preprocess": {
            "method": preprocess_result["method"],
            "deskew_angle": preprocess_result["deskew_angle"],
            "threshold_value": preprocess_result["threshold_value"],
        },
        "ocr_text": ocr_text,
        "cleaned_text": cleaned_text,
        "quality": quality,
        "prompt_preview": prompt[:180] + "...",
        "llm_output": llm_output,
        "retry_used": retry_used,
        "retry_prompt": retry_prompt,
        "retry_output": retry_output,
        "record": record,
        "error": error
    }

single_result = run_document_pipeline(documents[0])

pprint({k: v for k, v in single_result.items() if k not in ["ocr_text", "cleaned_text", "llm_output", "retry_output", "retry_prompt"]})

In [ ]:
pipeline_results = []

for doc in documents:
    result = run_document_pipeline(
        doc,
        simulate_broken_json=(doc["doc_id"] == "D003")
    )
    pipeline_results.append(result)

for result in pipeline_results:
    print("\nDocument:", result["doc_id"])
    print("Condition:", result["condition"])
    print("OCR quality:", result["quality"]["score"])
    print("Retry used:", result["retry_used"])
    print("Error:", result["error"])
    pprint(result["record"])

## 11. Create the final dataset

After validation, the records can be analyzed like normal data.

This is the output that business users and downstream systems usually need.

In [ ]:
records = [result["record"] for result in pipeline_results if result["error"] is None]
df = pd.DataFrame(records)

df

In [ ]:
invoice_df = df[df["doc_type"] == "invoice"].copy()
receipt_df = df[df["doc_type"] == "receipt"].copy()
delivery_df = df[df["doc_type"] == "delivery_note"].copy()

if not invoice_df.empty:
    invoice_df["conversion_rate"] = invoice_df["conversions"] / invoice_df["clicks"]
    invoice_df["spend_matches_total"] = abs(invoice_df["spend_eur"] - invoice_df["total_eur"]) < 0.01

invoice_df

In [ ]:
project_summary = {
    "documents_processed": len(pipeline_results),
    "successful_records": int(sum(result["error"] is None for result in pipeline_results)),
    "failed_records": int(sum(result["error"] is not None for result in pipeline_results)),
    "retry_count": int(sum(result["retry_used"] for result in pipeline_results)),
    "avg_ocr_quality": round(float(np.mean([result["quality"]["score"] for result in pipeline_results])), 3),
    "document_types": df["doc_type"].value_counts().to_dict(),
}

pprint(project_summary)

## 12. Field-level quality checks

A document pipeline should not stop at JSON validation.

Add business checks that match the document type.

In [ ]:
def check_record_quality(record):
    issues = []
    doc_type = record["doc_type"]

    if record["document_id"] is None:
        issues.append("Missing document_id")

    if record["date"] is None:
        issues.append("Missing date")

    if doc_type == "invoice":
        if record["spend_eur"] is None or record["total_eur"] is None:
            issues.append("Invoice missing spend or total")
        elif abs(record["spend_eur"] - record["total_eur"]) > 0.01:
            issues.append("Invoice spend does not match total")

        if record["clicks"] is None or record["conversions"] is None:
            issues.append("Invoice missing campaign metrics")

    if doc_type == "receipt":
        if record["item_total_eur"] is None or record["tax_eur"] is None or record["total_eur"] is None:
            issues.append("Receipt missing money fields")
        elif abs((record["item_total_eur"] + record["tax_eur"]) - record["total_eur"]) > 0.01:
            issues.append("Receipt total does not match item total plus tax")

    if doc_type == "delivery_note":
        if record["packages"] is None or record["delayed_packages"] is None:
            issues.append("Delivery note missing package counts")
        elif record["delayed_packages"] > record["packages"]:
            issues.append("Delayed packages greater than packages")

    return issues

quality_report = []

for record in records:
    issues = check_record_quality(record)
    quality_report.append({
        "doc_id": record["doc_id"],
        "doc_type": record["doc_type"],
        "issue_count": len(issues),
        "issues": issues
    })

quality_df = pd.DataFrame(quality_report)
quality_df

In [ ]:
assert quality_df["issue_count"].sum() == 0
print("All field-level quality checks passed.")

## 13. Create a final project report

A good pipeline report is short and useful.

It should show success rate, retries, quality, and any failed documents.

In [ ]:
def make_final_project_report(pipeline_results, quality_df):
    total = len(pipeline_results)
    successful = sum(result["error"] is None for result in pipeline_results)
    failed = total - successful
    retry_count = sum(result["retry_used"] for result in pipeline_results)
    avg_quality = np.mean([result["quality"]["score"] for result in pipeline_results])

    failed_docs = [
        {
            "doc_id": result["doc_id"],
            "error": result["error"]
        }
        for result in pipeline_results
        if result["error"] is not None
    ]

    issue_rows = quality_df[quality_df["issue_count"] > 0]

    return {
        "total_documents": total,
        "successful_documents": successful,
        "failed_documents": failed,
        "success_rate": round(successful / total, 3),
        "retry_count": int(retry_count),
        "average_ocr_quality": round(float(avg_quality), 3),
        "documents_with_quality_issues": int(len(issue_rows)),
        "failed_doc_details": failed_docs
    }

final_report = make_final_project_report(pipeline_results, quality_df)
pprint(final_report)

## 14. Save-ready outputs

In real work, you may save the final records as CSV, JSON, or database rows.

Here we only create the strings so the notebook does not depend on external files.

In [ ]:
records_json = json.dumps(records, indent=2)
records_csv_preview = df.to_csv(index=False)

show("JSON output preview", records_json[:800] + "...")
show("CSV output preview", records_csv_preview[:800] + "...")

## Tricky bits

Document intelligence projects fail when one stage is hidden.

Keep image preprocessing, OCR text, LLM output, validation errors, and final records visible during development.

In [ ]:
debug_table = pd.DataFrame([
    {
        "stage": "Image",
        "debug_artifact": "Original and preprocessed image",
        "common_problem": "Blur, skew, shadow, low resolution"
    },
    {
        "stage": "OCR",
        "debug_artifact": "Raw OCR text",
        "common_problem": "Wrong characters or missing words"
    },
    {
        "stage": "Cleanup",
        "debug_artifact": "Cleaned OCR text",
        "common_problem": "Over-repairing text"
    },
    {
        "stage": "LLM",
        "debug_artifact": "Prompt and raw model output",
        "common_problem": "Invalid JSON or missing fields"
    },
    {
        "stage": "Validation",
        "debug_artifact": "Validation error message",
        "common_problem": "Wrong type or failed business rule"
    },
])

debug_table

In [ ]:
def diagnose_result(result):
    if result["quality"]["score"] < 0.4:
        return "OCR quality is low. Check image preprocessing."

    if result["error"] is not None:
        return "Validation failed. Check LLM output and schema."

    if result["retry_used"]:
        return "Succeeded after retry. Inspect the first output."

    return "Healthy result."

for result in pipeline_results:
    print(result["doc_id"], "=>", diagnose_result(result))

## Trick questions

1. Why keep raw OCR text after extraction?

<details>
<summary>Answer</summary>

It helps debug mistakes and gives traceability for the structured fields.

</details>

2. Why validate after the LLM returns JSON?

<details>
<summary>Answer</summary>

JSON can be syntactically valid but still contain wrong types, missing fields, or impossible values.

</details>

3. Why use different quality checks for invoices, receipts, and delivery notes?

<details>
<summary>Answer</summary>

Each document type has different business rules and required fields.

</details>

4. Should the pipeline silently fix every failed record?

<details>
<summary>Answer</summary>

No. Repair small formatting issues, retry when useful, and send hard failures to manual review.

</details>

5. What is the main output of a document intelligence pipeline?

<details>
<summary>Answer</summary>

Clean, validated, structured data that can be used by analytics, automation, or downstream systems.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Create a document image for the first document.

img = ___

assert img is not None
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Preprocess document D002 using its condition.

prep = ___

assert isinstance(prep, dict)
assert "image" in prep
assert prep["method"] == "denoise + adaptive"
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Run OCR for D002.

ocr_text = ___

assert isinstance(ocr_text, str)
assert len(ocr_text) > 20
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Repair OCR errors in the text.

fixed_text = ___

assert "DOCUMENT TYPE" in fixed_text
assert "Total" in fixed_text
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Build an extraction prompt for D002.

prompt = ___

assert "Return valid JSON only" in prompt
assert "D002" in prompt
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Extract JSON with the mock LLM and parse it.

llm_output = mock_llm_extract("D002", fixed_text)
data, error = ___

assert error is None
assert data["doc_id"] == "D002"
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Validate the parsed record.

validated = ___

assert model_to_dict(validated)["doc_type"] == "receipt"
print("Exercise 7 passed.")

In [ ]:
# Exercise 8
# Run the full document pipeline for D003.

result = ___

assert result["doc_id"] == "D003"
assert result["error"] is None
print("Exercise 8 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
img = create_document_image(documents[0]["text"])
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
prep = preprocess_image(document_images["D002"], condition="noisy")
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
ocr_text = run_ocr(prep["image"], "D002")
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
fixed_text = repair_common_ocr_errors(ocr_text)
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
prompt = build_extraction_prompt("D002", fixed_text, target_schema)
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
data, error = safe_json_loads(llm_output)
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
validated = validate_record(data)
```

</details>

<details>
<summary>Exercise 8 solution</summary>

```python
result = run_document_pipeline(documents[2])
```

</details>

## Cumulative review exercises

These mix topics from Days 15 to 24. Fill in `___` and run each cell.

In [ ]:
# Review 1: Complaint classification
# Create a simple label list.

labels = ___

assert isinstance(labels, list)
assert len(labels) >= 3
print("Review 1 passed.")

In [ ]:
# Review 2: OpenAI API
# Fill the standard chat roles.

roles = ___

assert roles == ["system", "user", "assistant"]
print("Review 2 passed.")

In [ ]:
# Review 3: Ollama
# Fill the default local generate endpoint.

ollama_url = ___

assert ollama_url == "http://localhost:11434/api/generate"
print("Review 3 passed.")

In [ ]:
# Review 4: Prompt engineering
# Choose the prompting style with no examples.

prompt_style = ___

assert prompt_style.lower() == "zero-shot"
print("Review 4 passed.")

In [ ]:
# Review 5: Structured output
# Parse JSON text.

json_text = '{"doc_id": "D001", "total_eur": 100}'
parsed = ___

assert parsed["total_eur"] == 100
print("Review 5 passed.")

In [ ]:
# Review 6: Information extraction
# Calculate conversion rate.

record = {"clicks": 1000, "conversions": 80}
conversion_rate = ___

assert abs(conversion_rate - 0.08) < 1e-9
print("Review 6 passed.")

In [ ]:
# Review 7: Tesseract basics
# Choose the German language code.

german_lang = ___

assert german_lang == "deu"
print("Review 7 passed.")

In [ ]:
# Review 8: EasyOCR
# Create a language list for English and Turkish.

easyocr_languages = ___

assert easyocr_languages == ["en", "tr"] or easyocr_languages == ["tr", "en"]
print("Review 8 passed.")

In [ ]:
# Review 9: OpenCV preprocessing
# Convert D001 image to grayscale.

review_gray = ___

assert review_gray is not None
assert len(review_gray.shape) == 2
print("Review 9 passed.")

In [ ]:
# Review 10: OCR plus LLM pipeline
# Build an extraction prompt from OCR text.

review_prompt = ___

assert "JSON schema" in review_prompt
assert "Return valid JSON only" in review_prompt
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
labels = ["billing", "delivery", "technical"]

# Review 2
roles = ["system", "user", "assistant"]

# Review 3
ollama_url = "http://localhost:11434/api/generate"

# Review 4
prompt_style = "zero-shot"

# Review 5
parsed = json.loads(json_text)

# Review 6
conversion_rate = record["conversions"] / record["clicks"]

# Review 7
german_lang = "deu"

# Review 8
easyocr_languages = ["en", "tr"]

# Review 9
review_gray = to_grayscale(document_images["D001"])

# Review 10
review_prompt = build_extraction_prompt("D001", ocr_outputs["D001"], target_schema)
```

</details>

In [ ]:
cheat_sheet = '''
DAY 25 CHEAT SHEET: DOCUMENT INTELLIGENCE PROJECT

End-to-end pipeline:
1. Create or load document images.
2. Preprocess each image based on condition.
3. Run OCR.
4. Clean OCR text and repair common mistakes.
5. Build a strict extraction prompt.
6. Extract JSON with an LLM or mock extractor.
7. Parse and repair JSON.
8. Validate schema and business rules.
9. Run field-level quality checks.
10. Create final dataset and project report.

Preprocessing choices:
- Clean document: denoise + Otsu threshold
- Noisy document: denoise + adaptive threshold
- Skewed document: deskew + denoise + Otsu threshold

Validation checks:
- Correct document type
- Valid date format
- Non-negative numeric fields
- conversions <= clicks
- delayed_packages <= packages
- receipt item total + tax equals total
- invoice spend equals total

Debug artifacts:
- Preprocessed image
- Raw OCR text
- Cleaned OCR text
- LLM prompt
- Raw JSON output
- Validation errors
- Final record
'''

print(cheat_sheet)

## Next up: Day 26 — EmbeddingsDeepDive

You will start Phase 6 and learn OpenAI embeddings, sentence-transformers, and cosine similarity.